In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [2]:
df_2019 = pd.read_parquet("dados limpos/dados2019_limpos")
df_2020 = pd.read_parquet("dados limpos/dados2020_limpos")
df_2021 = pd.read_parquet("dados limpos/dados2021_limpos")
df_2022 = pd.read_parquet("dados limpos/dados2022_limpos")

In [3]:
df_2019["Ano"] = 2019
df_2020["Ano"] = 2020
df_2021["Ano"] = 2021
df_2022["Ano"] = 2022

In [4]:
df = pd.concat([df_2019, df_2020, df_2021, df_2022], axis = 0)

In [5]:
df.columns

Index(['TP_FAIXA_ETARIA', 'TP_COR_RACA', 'TP_ST_CONCLUSAO', 'TP_ESCOLA',
       'TP_ENSINO', 'IN_TREINEIRO', 'CO_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC',
       'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'TP_PRESENCA_CN',
       'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'NU_NOTA_CN',
       'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'TX_RESPOSTAS_CN',
       'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC', 'TX_RESPOSTAS_MT',
       'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC', 'TX_GABARITO_MT',
       'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3',
       'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO', 'Q001', 'Q002',
       'Q003', 'Q004', 'Q006', 'Q022', 'Q024', 'Q025', 'Ano'],
      dtype='object')

In [6]:
df

,TP_FAIXA_ETARIA,TP_COR_RACA,TP_ST_CONCLUSAO,TP_ESCOLA,TP_ENSINO,IN_TREINEIRO,CO_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,...,NU_NOTA_REDACAO,Q001,Q002,Q003,Q004,Q006,Q022,Q024,Q025,Ano
index,,,,,,,,,,,,,,,,,,,,,
26,2,1,2,3,1,0,41,4,1,1,...,900,E,E,B,B,E,C,B,B,2019
28,3,1,2,2,1,0,43,2,1,1,...,580,E,B,B,B,A,D,B,A,2019
31,2,2,2,2,1,0,35,2,1,1,...,660,E,F,B,D,B,D,A,B,2019
34,2,2,2,2,1,0,29,2,1,1,...,620,E,C,B,B,B,C,A,B,2019
53,3,3,2,2,1,0,25,2,1,1,...,480,D,E,C,B,C,E,A,B,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3476074,3,4,2,2,1,0,28,2,1,1,...,700,E,E,B,B,D,E,B,B,2022
3476077,2,3,2,2,1,0,31,4,1,1,...,820,F,F,D,D,H,E,D,B,2022
3476090,3,1,2,3,1,0,21,4,1,1,...,820,G,F,E,D,O,D,B,B,2022


In [7]:
df['TX_RESPOSTAS_TOTAL'] = (
    
    df['TX_RESPOSTAS_LC'].fillna('') +
    df['TX_RESPOSTAS_CH'].fillna('') + 
    df['TX_RESPOSTAS_CN'].fillna('') + 
    df['TX_RESPOSTAS_MT'].fillna('')
)

df['TX_GABARITOS_TOTAL'] = (
    df['TX_GABARITO_LC'].fillna('') + 
    df['TX_GABARITO_CH'].fillna('') + 
    df['TX_GABARITO_CN'].fillna('') + 
    df['TX_GABARITO_MT'].fillna('')
)

In [8]:
def matriz_acertos(col_resp, col_gab):

    n = len(col_resp)

    M = np.zeros((n,180), dtype=np.int8)

    for i, (r, g) in enumerate(zip(col_resp, col_gab)):

        idx = 0

        for ri, gi in zip(r, g):

            if ri == '9':
                continue

            if ri in 'ABCDE':
                M[i, idx] = (ri == gi)

            idx += 1

    return M

In [9]:
matriz_acertos =matriz_acertos(df['TX_RESPOSTAS_TOTAL'], df['TX_GABARITOS_TOTAL'])

In [10]:
cols = [f'Q{i+1}' for i in range(180)]

df_acertos = pd.DataFrame(matriz_acertos, columns=cols)

df = df.reset_index(drop=True)
df_acertos = df_acertos.reset_index(drop=True)

df = pd.concat([df, df_acertos], axis=1)

In [ ]:
df.to_csv("DadosENEM_2019-2022.csv")

In [18]:
os.makedirs("dadosENEM_2019-2022_parquet", exist_ok=True)

chunks = pd.read_csv("DadosENEM_2019-2022.csv", chunksize=100000)

for i, chunk in enumerate(chunks):
    chunk.to_parquet(f"dadosENEM_2019-2022_parquet/parte_{i}.parquet")